# 第 2 课｜RAG 不只是向量数据库

**核心问题：** 资料已经存在，为什么模型仍然可能用错？

本 Notebook 完全离线。我们从租户 A 的退款政策开始，逐步观察：资料如何被找到、如何被挡住、哪些内容真正进入上下文、结论怎样回到来源，以及资料不足时程序为什么应该停下。

本课的实现是教学实验，不是生产 embedding 或向量数据库。每个中间结果都保留下来，方便定位错误发生在检索、过滤、装配还是引用阶段。


## 运行约定

- 从空 kernel 按顺序运行；不需要网络、在线模型、API key 或外部数据库。
- 标识符、字段和运行记录使用 English；解释文字使用中文。
- 本课前半段手写 BM25、教学型向量表示、RRF、过滤、装配和来源检查。
- hash_embedding / teaching_embedding 只是透明的离线替身，不代表真实语义模型。


In [ ]:
from datetime import date
from pathlib import Path

from projects.capstones.research_agent.teaching import (
    GroundedDocument,
    GroundedRequest,
    OfflineKnowledgeAssistant,
    assemble_context,
    assemble_context_text,
    choose_context_mode,
    decompose_question,
    dense_rank,
    estimate_tokens,
    filter_candidates,
    lexical_rank,
    lexical_score,
    load_grounded_corpus,
    reciprocal_rank_fusion,
    verify_citations,
)

CORPUS_PATH = Path("projects/capstones/research_agent/fixtures/tenant_policy_corpus.jsonl")
documents = load_grounded_corpus(CORPUS_PATH)
documents_by_id = {document.source_id: document for document in documents}
request = GroundedRequest(
    question="Tenant A refund policy current processing time",
    tenant_id="A",
    as_of=date(2026, 8, 13),
    claim_keys=("refund_current",),
    token_budget=900,
    top_k=8,
)
print(f"documents={len(documents)}")
print(f"question={request.question}")
print(f"tenant={request.tenant_id}, as_of={request.as_of}, budget={request.token_budget}")


## 1. 先做选择：全文输入，还是检索后装配？

如果小语料完整放得下，全文输入是一个值得先测的简单基线；如果语料超过预算，就需要检索后装配。不要默认“有 RAG 才先进”，先看输入规模和交付边界。


In [ ]:
mode = choose_context_mode(documents, request.token_budget)
print(mode)
print("corpus_token_estimate:", mode.corpus_tokens)


In [ ]:
full_context = "\n\n".join(
    f"[{document.source_id}] {document.contextual_text}"
    for document in documents
)
print("full_context_token_estimate:", estimate_tokens(full_context))
print("full_context_is_within_budget:", estimate_tokens(full_context) <= request.token_budget)


## 2. 词法检索：先看精确词项为什么有价值

BM25 不是一个神秘的分数。它把词频、词项稀有度和文档长度放在一起，回答的是“哪些文档在词面上更像 query”。它还没有回答租户、日期或来源是否适用。


In [ ]:
def student_bm25_score(
    query: str,
    document: GroundedDocument,
    corpus: list[GroundedDocument],
) -> float:
    # TODO: count query/document token overlap, then add IDF and length normalization.
    # Do not call lexical_score here; the goal is to reproduce the mechanism.
    raise NotImplementedError("手写 BM25：词频、稀有度、长度归一化")


In [ ]:
lexical_results = sorted(
    ((document.source_id, round(student_bm25_score(request.question, document, documents), 4))
     for document in documents),
    key=lambda item: (-item[1], item[0]),
)
for rank, item in enumerate(lexical_results[:8], 1):
    print(rank, item)


## 3. 语义型检索：词面不同，也要有机会遇到同一资料

真实系统通常使用训练得到的 embedding。为了保持本 Notebook 离线，我们使用可见的别名扩展和确定性特征向量模拟“意义接近”的信号。它的价值是帮助观察接口，不是替代真实模型。


In [ ]:
def student_dense_rank(query: str, corpus: list[GroundedDocument]) -> list[tuple[str, float]]:
    # TODO: create a fixed vector for the query and each document, then sort cosine scores.
    # Keep this implementation deterministic and explain the feature choice.
    raise NotImplementedError("手写教学型稠密排序")


In [ ]:
dense_results = student_dense_rank("取消订单后的钱什么时候回来", documents)
for rank, item in enumerate(dense_results[:8], 1):
    print(rank, item[0], round(item[1], 4))


## 4. 混合检索：两套排名互补，先用 RRF 合并名次

BM25 对编号和精确术语有优势，语义型检索对改写问法有帮助。RRF 只融合名次，不把两种分数假装成同一个概率。


In [ ]:
def student_rrf(
    lexical: list[tuple[str, float]],
    dense: list[tuple[str, float]],
    k: int = 60,
):
    # TODO: build one-based rank maps and sum 1 / (k + rank) for each source.
    raise NotImplementedError("手写 RRF 名次融合")


In [ ]:
lexical = lexical_rank(request.question, documents)
dense = dense_rank(request.question, documents)
hybrid_hits = student_rrf(lexical, dense)
for rank, hit in enumerate(hybrid_hits[:8], 1):
    print(rank, hit.source_id, round(hit.score, 6), hit.lexical_rank, hit.dense_rank)


## 5. 过滤：候选不是允许资料

排名只说明“可能相关”。进入模型输入之前，还要检查租户、日期和文档内容边界。文档里的“忽略规则”仍然只是文档文字，不能改变程序控制条件。


In [ ]:
def student_filter(
    hits,
    documents_by_id: dict[str, GroundedDocument],
    request: GroundedRequest,
):
    # TODO: return one decision per hit.
    # Required reasons: tenant_mismatch, expired_or_not_yet_effective,
    # untrusted_instruction, allowed.
    raise NotImplementedError("手写过滤规则")


In [ ]:
decisions = student_filter(hybrid_hits[: request.top_k], documents_by_id, request)
for decision in decisions:
    print(decision.source_id, decision.allowed, decision.reason)


## 6. 装配：允许资料还要通过输入预算

装配器只接收允许资料，并按照预算决定哪些片段真正进入上下文。被舍弃的资料仍然保留原因，方便回看“它是没找到，还是找到后没被带回”。


In [ ]:
def student_assemble(
    decisions,
    documents_by_id: dict[str, GroundedDocument],
    request: GroundedRequest,
):
    # TODO: walk allowed decisions, estimate cost, and keep selected/discarded IDs.
    # Every discard must include a trace reason.
    raise NotImplementedError("手写上下文装配")


In [ ]:
selected_ids, discarded_ids, assembly_trace = student_assemble(
    decisions, documents_by_id, request
)
print("selected:", selected_ids)
print("discarded:", discarded_ids)
print("context:\n", assemble_context_text(selected_ids, documents_by_id))
print("trace:")
for event in assembly_trace:
    print(event)


## 7. 引用：结论必须能走回被选中的来源

一个引用 ID 只是起点。至少要检查它确实属于 selected context；课程项目还要进一步检查源文本是否包含支持该主张的 statement。


In [ ]:
def student_citation_check(citations: dict[str, list[str]], selected: list[str]) -> list[str]:
    # TODO: report empty source lists and citations that point outside selected.
    raise NotImplementedError("手写引用映射检查")


In [ ]:
from projects.capstones.research_agent.teaching import build_claims

citations, conflicts, missing = build_claims(request.claim_keys, selected_ids, documents_by_id)
print("citations:", citations)
print("conflicts:", conflicts)
print("missing:", missing)
print("citation_failures:", student_citation_check(citations, selected_ids))


## 8. 正常运行：把手写部件组合成完整助手

下面的 OfflineKnowledgeAssistant 是教师完整例程。学生版本要完成前面的 TODO 后，再比较每层输出，而不是只看最后一句答案。


## 8.5 课后探索：VideoRAG 把来源扩展成带时间码的片段

视频资料除了字幕，还可能有白板、动作和版面等视觉线索。这里不下载或解码真实视频，而是使用本地 shot metadata，观察同一条“候选—允许—上下文—引用—停止”路径怎样增加时间和许可证边界。


## 9. 多跳问题：先分解，再分别寻找资料

多跳问题不是把一个更长的 query 丢给同一个检索器。先把问题拆成子问题，分别找到政策、执行路径和团队资料，再合并检查结果。


## 10. 失败路径：资料不足、冲突和预算不足都要有名字

程序不是只有 completed 一个终态。明确的停止原因让我们知道下一步应该补资料、调整检索、解决权限，还是请人确认冲突。


## 运行结果可视化：候选资料怎样变成可引用证据

图中的 source ID 来自刚才同一次运行。请比较正常、冲突和资料不足三种状态；不要把“检索到候选”误当成“主张已经得到支持”。


## 11. Mini Project 交付

完成学生 Notebook 后，提交一条正常路径和一条失败路径。展示时必须指出：

- 哪些资料只是 candidate；
- 哪些资料在 filter 阶段被挡住；
- 哪些 source_id 真正进入 context；
- 哪个 claim 指向哪个来源；
- 程序为什么 completed、conflict_detected 或 insufficient_evidence。

本课的核心不是“接上一个框架”，而是能从运行记录解释信息为什么被使用或拒绝。
